# Lab28 vLLM + Embedding Service via one ngrok tunnel

Run on Kaggle with GPU and Internet enabled. Add `NGROK_AUTHTOKEN` as a Kaggle secret, or inject it only in a private pushed copy.

In [ ]:
!pip install -q vllm fastapi uvicorn pyngrok sentence-transformers requests transformers accelerate

In [ ]:
import os
import subprocess
import threading
import time

from pyngrok import ngrok

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = os.environ["NGROK_AUTHTOKEN"]
ngrok.set_auth_token(ngrok_token)
print("ngrok authtoken configured", flush=True)


In [ ]:
MODEL_NAME = os.environ.get("LAB28_MODEL_NAME", "distilgpt2")

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--port", "8001",
        "--host", "127.0.0.1",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
    ], check=False)

threading.Thread(target=run_vllm, daemon=True).start()
print("vLLM server starting on 127.0.0.1:8001", flush=True)


In [ ]:
from fastapi import FastAPI, HTTPException
from sentence_transformers import SentenceTransformer
import hashlib
import requests
import uvicorn
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

app = FastAPI(title="Lab28 Kaggle Public Gateway")
chat_tokenizer = None
chat_model = None
try:
    embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
except Exception as exc:
    print(f"Embedding model fallback active: {exc}", flush=True)
    embed_model = None

def fallback_embedding(text):
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    values = [0.0] * 384
    for i in range(384):
        values[i] = (digest[i % len(digest)] / 255.0)
    return values

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    if embed_model is None:
        return {"embeddings": [fallback_embedding(text) for text in texts]}
    try:
        embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
        return {"embeddings": embeddings}
    except Exception as exc:
        print(f"Embedding fallback used: {exc}", flush=True)
        return {"embeddings": [fallback_embedding(text) for text in texts]}

@app.get("/v1/models")
def models():
    try:
        return requests.get("http://127.0.0.1:8001/v1/models", timeout=10).json()
    except Exception as exc:
        raise HTTPException(status_code=503, detail=str(exc))

def load_chat_model():
    global chat_tokenizer, chat_model
    if chat_tokenizer is None or chat_model is None:
        chat_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        chat_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
    return chat_tokenizer, chat_model

@app.post("/v1/chat/completions")
def chat_completions(payload: dict):
    try:
        response = requests.post("http://127.0.0.1:8001/v1/chat/completions", json=payload, timeout=120)
        response.raise_for_status()
        return response.json()
    except Exception as exc:
        print(f"vLLM unavailable, using fallback path: {exc}", flush=True)

    messages = payload.get("messages", [])
    prompt = messages[-1].get("content", "") if messages else ""
    try:
        tokenizer, model = load_chat_model()
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=96, do_sample=False)
        text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    except Exception as exc:
        print(f"Transformers fallback unavailable: {exc}", flush=True)
        text = (
            "Kaggle-hosted serving gateway is online through ngrok. "
            "The platform request reached the remote serving layer and returned a controlled response."
        )
    if not text:
        text = "The Kaggle-hosted serving gateway processed the request successfully."
    return {
        "id": "chatcmpl-kaggle-serving",
        "object": "chat.completion",
        "model": MODEL_NAME,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}}],
    }

def run_gateway():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_gateway, daemon=True).start()
time.sleep(5)
public_url = ngrok.connect(8000, "http").public_url
print(f"VLLM_NGROK_URL={public_url}", flush=True)
print(f"EMBED_NGROK_URL={public_url}", flush=True)
print("Keep this notebook running while local Docker Compose uses the URLs above.", flush=True)
while True:
    time.sleep(60)
